In [4]:
import pandas as pd

df = pd.read_excel(r"C:\Users\LENOVO\Downloads\online+retail\Online Retail.xlsx")
df.head(), df.shape


(  InvoiceNo StockCode                          Description  Quantity  \
 0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
 1    536365     71053                  WHITE METAL LANTERN         6   
 2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
 3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
 4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   
 
           InvoiceDate  UnitPrice  CustomerID         Country  
 0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
 1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
 2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
 3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
 4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  ,
 (541909, 8))

In [5]:

df = df.dropna(subset=['CustomerID'])

df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

df.shape


(397924, 8)

In [6]:
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

df[['Quantity', 'UnitPrice', 'TotalPrice']].head()


,Quantity,UnitPrice,TotalPrice
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34


In [ ]:
import datetime as dt

reference_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (reference_date - x.max()).days,  # Recency
    'InvoiceNo': 'nunique',                                     # Frequency
    'TotalPrice': 'sum'                                         # Monetary
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']

rfm.head()


,Recency,Frequency,Monetary
CustomerID,,,
12346.0,326,1,77183.60
12347.0,2,7,4310.00
12348.0,75,4,1797.24
12349.0,19,1,1757.55
12350.0,310,1,334.40


In [ ]:
rfm['R_Score'] = pd.qcut(
    rfm['Recency'],
    4,
    labels=[4, 3, 2, 1]
)

rfm['F_Score'] = pd.cut(
    rfm['Frequency'],
    bins=4,
    labels=[1, 2, 3, 4]
)

rfm['M_Score'] = pd.cut(
    rfm['Monetary'],
    bins=4,
    labels=[1, 2, 3, 4]
)

rfm[['Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score']].head()





,Recency,Frequency,Monetary,R_Score,F_Score,M_Score
CustomerID,,,,,,
12346.0,326,1,77183.60,1,1,2
12347.0,2,7,4310.00,4,1,1
12348.0,75,4,1797.24,2,1,1
12349.0,19,1,1757.55,3,1,1
12350.0,310,1,334.40,1,1,1


In [ ]:
rfm['RFM_Score'] = (
    rfm['R_Score'].astype(str) +
    rfm['F_Score'].astype(str) +
    rfm['M_Score'].astype(str)
)

def segment_customer(row):
    if row['R_Score'] == 4 and row['F_Score'] == 4:
        return 'Champions'
    elif row['R_Score'] >= 3 and row['F_Score'] >= 3:
        return 'Loyal Customers'
    elif row['R_Score'] >= 3 and row['F_Score'] <= 2:
        return 'Potential Loyalist'
    elif row['R_Score'] <= 2 and row['F_Score'] >= 3:
        return 'At Risk'
    else:
        return 'Hibernating'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)

rfm[['RFM_Score', 'Segment']].head()



,RFM_Score,Segment
CustomerID,,
12346.0,114,Hibernating
12347.0,444,Champions
12348.0,234,At Risk
12349.0,314,Potential Loyalist
12350.0,112,Hibernating


In [ ]:
segment_summary = rfm.groupby('Segment').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': ['mean', 'count']
}).round(1)

print(segment_summary)

                   Recency Frequency Monetary      
                      mean      mean     mean count
Segment                                            
At Risk              118.3       4.2   1611.0   655
Champions              7.4      13.9   7659.8   591
Hibernating          185.7       1.3    510.5  1514
Loyal Customers       24.1       5.1   2232.1   923
Potential Loyalist    24.8       1.5    756.4   656


In [14]:
rfm.to_csv("rfm_final.csv", index=False)


In [20]:
def my_function(x):
    return x + 1
print(my_function(24))

25
